In [1]:
import json
import pandas as pd
from tqdm import tqdm 
from model import *
from prompt import *

tqdm.pandas()

In [2]:
def parse_llm_response(response):
    """
    Safely parse LLM response to ensure valid JSON.
    Returns default_value if parsing fails.
    """
    if "```json" in response:
        try:
            start_idx = response.index("```json") + 7
            end_idx = response.index("```", start_idx)
            response = response[start_idx:end_idx].strip()
        except ValueError:
            pass
    elif "```" in response:
        try:
            start_idx = response.index("```") + 3
            end_idx = response.index("```", start_idx)
            response = response[start_idx:end_idx].strip()
        except ValueError:
            pass
            
    try:
        return json.loads(response)
    except json.JSONDecodeError as e:
        print(f"Failed to parse LLM response as JSON: {e}")
        print("Response was:", response[:200], "..." if len(response) > 200 else "")

### Examples

In [3]:
example1 = """
• Louisiana extends 3 pre-1954 U.S nautical miles (3.455 miles or 5.560 kilometers)
seaward.
• Texas and the Florida Gulf Coast extend 9 U.S. nautical miles (10.4 miles or 16.7
kilometers) seaward.
"""

example2 = """
    A wide variety of different radar systems exist and have been developed for wave measurements. The most
common operate in the microwave radio-frequency band, and use a continuous wave frequency modulated
or pulsed signal in the GHz range. These systems can generally be separated into two categories,
downward-facing and forward-facing radars.
"""

In [6]:
input_prompt1 = prompt.format(DOCUMENTATION = example1)
response1 = call_gpt4o_mini(input_prompt1)
parse_llm_response(response1)

{'document_metadata': {'title': 'Offshore Wind Farm Regulatory Overview',
  'document_number': None,
  'Type of wind farm': 'off-shore'},
 'regulatory_entities': [{'entity_name': 'Louisiana State Government',
   'jurisdiction': 'state',
   'role': 'admin'},
  {'entity_name': 'Texas State Government',
   'jurisdiction': 'state',
   'role': 'admin'},
  {'entity_name': 'Florida State Government',
   'jurisdiction': 'state',
   'role': 'admin'}],
 'regulatory_constraints': [{'type': 'Spatial',
   'requirement': 'Louisiana extends 3 pre-1954 U.S nautical miles (3.455 miles or 5.560 kilometers) seaward.',
   'scope': 'Louisiana coastal waters',
   'numerical_value': '3',
   'unit': 'nautical miles',
   'source': 'Louisiana State Regulations',
   'related_domains': 'spatial planning'},
  {'type': 'Spatial',
   'requirement': 'Texas and the Florida Gulf Coast extend 9 U.S. nautical miles (10.4 miles or 16.7 kilometers) seaward.',
   'scope': 'Texas and Florida Gulf Coast waters',
   'numerical

In [4]:
input_prompt2 = prompt.format(DOCUMENTATION = example2)
response2 = call_gpt4o_mini(input_prompt2)
parse_llm_response(response2)

{'document_metadata': {'title': 'Radar Systems for Wave Measurements',
  'document_number': None,
  'Type of wind farm': 'off-shore'},
 'regulatory_entities': None,
 'regulatory_constraints': None}

### Scaling to the dataset

In [8]:
df = pd.read_csv('Specs/21.csv')
df


,document_id,page_number,content
0,21,1,Metocean Characterization Recommended Practice...
1,21,2,"DNV GL – Document No.: 10039663-HOU-01, Issue:..."
2,21,3,"DNV GL – Document No.: 10039663-HOU-01, Issue:..."
3,21,4,"DNV GL – Document No.: 10039663-HOU-01, Issue:..."
4,21,5,"DNV GL – Document No.: 10039663-HOU-01, Issue:..."
...,...,...,...
125,21,126,"DNV GL – Document No.: 10039663-HOU-01, Issue:..."
126,21,127,"DNV GL – Document No.: 10039663-HOU-01, Issue:..."
127,21,128,"DNV GL – Document No.: 10039663-HOU-01, Issue:..."
128,21,129,"DNV GL – Document No.: 10039663-HOU-01, Issue:..."


In [9]:
df['content'].iloc[1]

'DNV GL – Document No.: 10039663-HOU-01, Issue: D, Status: FINAL www.dnvgl.com Page 2 Acknowledgement Prepared under BOEM Award Contract No. M17PC00004 by DNV KEMA Renewables, Inc. (DNV GL). DNV GL would like to acknowledge and thank the following individuals for their valuable review and comments on this report: • Joel Cline, U.S. Department of Energy • Matthew Filippelli, UL • George Hagerman, Old Dominion University • Katrine Sønderbye Jensen, Ørsted • Anthony Kirincich, Woods Hole Oceanographic Institution • Nikolaj Kruppa, Ørsted • Will Shaw, Pacific Northwest National Laboratory • Erik Smid, Siemens Gamesa Renewable Energy • Niels Jacob Tarp-Johansen, Ørsted • Lorry Wagner, Lake Erie Energy Development Corporation (LEEDCO) Disclaimer This report was prepared under contract between the Bureau of Ocean Energy Management (BOEM) and DNV GL. This report has been technically reviewed by BOEM and has been approved for publication. Approval does not signify that the contents necessarily 

In [10]:
df['prompt'] = df['content'].apply(
        lambda content: prompt.format(DOCUMENTATION=content)
    )

df

,document_id,page_number,content,prompt
0,21,1,Metocean Characterization Recommended Practice...,\nYou are an expert system designed to extract...
1,21,2,"DNV GL – Document No.: 10039663-HOU-01, Issue:...",\nYou are an expert system designed to extract...
2,21,3,"DNV GL – Document No.: 10039663-HOU-01, Issue:...",\nYou are an expert system designed to extract...
3,21,4,"DNV GL – Document No.: 10039663-HOU-01, Issue:...",\nYou are an expert system designed to extract...
4,21,5,"DNV GL – Document No.: 10039663-HOU-01, Issue:...",\nYou are an expert system designed to extract...
...,...,...,...,...
125,21,126,"DNV GL – Document No.: 10039663-HOU-01, Issue:...",\nYou are an expert system designed to extract...
126,21,127,"DNV GL – Document No.: 10039663-HOU-01, Issue:...",\nYou are an expert system designed to extract...
127,21,128,"DNV GL – Document No.: 10039663-HOU-01, Issue:...",\nYou are an expert system designed to extract...
128,21,129,"DNV GL – Document No.: 10039663-HOU-01, Issue:...",\nYou are an expert system designed to extract...


In [ ]:
def safe_process(df, step=10, output="21_response.csv"):
    try:
        for i, row in df.iterrows():
            # Modify as needed
            df.at[i, 'gpt4o_mini_response'] = call_gpt4o_mini(df.at[i, 'prompt'])

            if i % step == 0:
                df.to_csv(output)
                print(f"Checkpoint saved at row {i}")
        
        df.to_csv("21_response.csv")
        print("Final save done.")
    except Exception as e:
      print(f"Exception occurred: {e}")
      df.to_csv("21_response.csv")
      print("Done.")
      raise

In [ ]:
#df['gpt4o_mini_response'] = df['prompt'].progress_apply(call_gpt4o_mini)
df

100%|██████████| 130/130 [14:41<00:00,  6.78s/it]


,document_id,page_number,content,prompt,gpt4o_mini_response
0,21,1,Metocean Characterization Recommended Practice...,\nYou are an expert system designed to extract...,"```json\n{\n ""document_metadata"": {\n ""tit..."
1,21,2,"DNV GL – Document No.: 10039663-HOU-01, Issue:...",\nYou are an expert system designed to extract...,"```json\n{\n ""document_metadata"": {\n ""tit..."
2,21,3,"DNV GL – Document No.: 10039663-HOU-01, Issue:...",\nYou are an expert system designed to extract...,"```json\n{\n ""document_metadata"": {\n ""tit..."
3,21,4,"DNV GL – Document No.: 10039663-HOU-01, Issue:...",\nYou are an expert system designed to extract...,"```json\n{\n ""document_metadata"": {\n ""tit..."
4,21,5,"DNV GL – Document No.: 10039663-HOU-01, Issue:...",\nYou are an expert system designed to extract...,"```json\n{\n ""document_metadata"": {\n ""tit..."
...,...,...,...,...,...
125,21,126,"DNV GL – Document No.: 10039663-HOU-01, Issue:...",\nYou are an expert system designed to extract...,"```json\n{\n ""document_metadata"": {\n ""tit..."
126,21,127,"DNV GL – Document No.: 10039663-HOU-01, Issue:...",\nYou are an expert system designed to extract...,"```json\n{\n ""document_metadata"": {\n ""tit..."
127,21,128,"DNV GL – Document No.: 10039663-HOU-01, Issue:...",\nYou are an expert system designed to extract...,"```json\n{\n ""document_metadata"": {\n ""tit..."
128,21,129,"DNV GL – Document No.: 10039663-HOU-01, Issue:...",\nYou are an expert system designed to extract...,"```json\n{\n ""document_metadata"": {\n ""tit..."


In [12]:
df['clean_response'] = df['gpt4o_mini_response'].progress_apply(parse_llm_response)
df

100%|██████████| 130/130 [00:00<00:00, 130040.43it/s]


,document_id,page_number,content,prompt,gpt4o_mini_response,clean_response
0,21,1,Metocean Characterization Recommended Practice...,\nYou are an expert system designed to extract...,"```json\n{\n ""document_metadata"": {\n ""tit...",{'document_metadata': {'title': 'Metocean Char...
1,21,2,"DNV GL – Document No.: 10039663-HOU-01, Issue:...",\nYou are an expert system designed to extract...,"```json\n{\n ""document_metadata"": {\n ""tit...",{'document_metadata': {'title': 'DNV GL Report...
2,21,3,"DNV GL – Document No.: 10039663-HOU-01, Issue:...",\nYou are an expert system designed to extract...,"```json\n{\n ""document_metadata"": {\n ""tit...",{'document_metadata': {'title': 'DNV GL – Docu...
3,21,4,"DNV GL – Document No.: 10039663-HOU-01, Issue:...",\nYou are an expert system designed to extract...,"```json\n{\n ""document_metadata"": {\n ""tit...",{'document_metadata': {'title': 'DNV GL – Docu...
4,21,5,"DNV GL – Document No.: 10039663-HOU-01, Issue:...",\nYou are an expert system designed to extract...,"```json\n{\n ""document_metadata"": {\n ""tit...",{'document_metadata': {'title': 'DNV GL – Docu...
...,...,...,...,...,...,...
125,21,126,"DNV GL – Document No.: 10039663-HOU-01, Issue:...",\nYou are an expert system designed to extract...,"```json\n{\n ""document_metadata"": {\n ""tit...",{'document_metadata': {'title': 'DNV GL – Docu...
126,21,127,"DNV GL – Document No.: 10039663-HOU-01, Issue:...",\nYou are an expert system designed to extract...,"```json\n{\n ""document_metadata"": {\n ""tit...",{'document_metadata': {'title': 'DNV GL – Docu...
127,21,128,"DNV GL – Document No.: 10039663-HOU-01, Issue:...",\nYou are an expert system designed to extract...,"```json\n{\n ""document_metadata"": {\n ""tit...",{'document_metadata': {'title': 'DNV GL – Docu...
128,21,129,"DNV GL – Document No.: 10039663-HOU-01, Issue:...",\nYou are an expert system designed to extract...,"```json\n{\n ""document_metadata"": {\n ""tit...",{'document_metadata': {'title': 'DNV GL – Docu...


In [13]:
df.to_csv("21_response.csv",index=False)